# 8.13 — Decoder ablation: is the sophisticated decoder necessary?

Reviewer-style critique: *there is no need for a BatchNorm + Dropout network decoder.* This notebook
re-runs the 8.12 block-split decode (same sampling / QC / bins) with simpler decoders and overlays them.

Note on softmax: the 8.12 decoder has **no softmax layer** — `nn.CrossEntropyLoss` applies log-softmax inside the
loss, and accuracy is `argmax` of the raw logits (softmax never changes the argmax). So "removing softmax" only
matters at the *loss* level: the `linear_mse` variant trains a plain linear readout on one-hot targets (no softmax).

| variant | normalisation | dropout | loss |
|---|---|---|---|
| `full` (8.12) | BatchNorm | 0.5 | CE |
| `no_dropout` | BatchNorm | - | CE |
| `no_bn` | none (raw rates) | 0.5 | CE |
| `plain_raw` | none | - | CE |
| `plain_z` | z-score (train stats) | - | CE (= plain multinomial logistic regression) |
| `linear_mse` | z-score (train stats) | - | MSE on one-hot (no softmax) |

Output: `processed/decoder/decoding_results_8.13.pkl`, keyed `[variant][block_condition][direction]`.


In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import pickle
import numpy as np
import torch
import torch.nn as nn

from imports import *  # Path, np, pd, plt
from config import dir_config, ephys_config
from src.utils import decoding_utils

In [ ]:
processed_dir = Path(dir_config.data.processed)
GP_CONFIG        = ephys_config["alignment_settings_GP"]
alignments    = list(GP_CONFIG.keys())          # ['baseline', 'visual', 'cue', 'response']
device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| alignments:", alignments)

session_to_exclude = ["210210_GP_JP", "241209_GP_TZ"]

session_metadata = pd.read_csv(processed_dir / "sessions_metadata.csv")
session_metadata = session_metadata[~session_metadata.session_id.isin(session_to_exclude)].reset_index(drop=True)

neuron_metadata = pd.read_csv(processed_dir / "neuron_metadata.csv")
neuron_metadata = neuron_metadata[~neuron_metadata.session_id.isin(session_to_exclude)].reset_index(drop=True)

with open(processed_dir / "glm_hmm_models" / "glm_hmm_masked_final.pkl", "rb") as f:
    glm_hmm = pickle.load(f)
# ephys_neuron_wise.pkl (3 GB) is loaded lazily in the decode cell, only when actually decoding

In [ ]:
# --- trial info (toRF reference frame; carries prior_direction) -> correct-only + abs_coherence ---
trial_info_all = decoding_utils.prepare_trial_info(session_metadata, glm_hmm)
print("prior_direction counts:", dict(trial_info_all.prior_direction.value_counts()))

# GUARD: choice == 1 must mean toRF. In biased blocks at 0% coherence, toRF sessions should
# choose toRF (>0.5) and awayRF sessions should choose away (<0.5).
def choice_bias(direction):
    subset = trial_info_all[(trial_info_all.prior_direction == direction)
                            & (trial_info_all.signed_coherence == 0)
                            & (trial_info_all.prior_block == 1)]
    return subset.choice.mean()
mean_choice_toRF, mean_choice_awayRF = choice_bias("toRF"), choice_bias("awayRF")
print(f"biased-block 0%-coh mean(choice=1/toRF): toRF={mean_choice_toRF:.3f} (>0.5)  awayRF={mean_choice_awayRF:.3f} (<0.5)")
assert mean_choice_toRF > 0.5 and mean_choice_awayRF < 0.5, "choice coding is NOT toRF=1 -- investigate before decoding!"

RT_TRIM_MS = 250  # reject fast trials so cue/response epochs reach the decision period; QC: nb 8.00
trial_info = trial_info_all[trial_info_all.outcome == 1].reset_index(drop=True)  # correct only
trial_info = trial_info[trial_info.reaction_time >= RT_TRIM_MS].reset_index(drop=True)
trial_info["abs_coherence"] = trial_info.signed_coherence.abs()
print(f"correct + RT>={RT_TRIM_MS}ms trials: {len(trial_info)}")
print("abs_coherence levels:", np.sort(trial_info.abs_coherence.unique()))
print("choice classes:", np.sort(trial_info.choice.unique()),
      "| hmm_state classes:", np.sort(trial_info.hmm_state.unique()))
print("prior_block counts (0=equal, 1=unequal):", dict(trial_info.prior_block.value_counts()))
print(trial_info.groupby(["prior_block", "prior_direction"]).size())


In [ ]:
# --- decoder variants ---
VARIANTS = {   # name: dict(bn, dropout, zscore, loss)
    "full":       dict(bn=True,  dropout=0.5,  zscore=False, loss="ce"),
    "no_dropout": dict(bn=True,  dropout=0.0,  zscore=False, loss="ce"),
    "no_bn":      dict(bn=False, dropout=0.5,  zscore=False, loss="ce"),
    "plain_raw":  dict(bn=False, dropout=0.0,  zscore=False, loss="ce"),
    "plain_z":    dict(bn=False, dropout=0.0,  zscore=True,  loss="ce"),
    "linear_mse": dict(bn=False, dropout=0.0,  zscore=True,  loss="mse"),
}

def build_decoder(n_input, n_classes, bn, dropout):
    layers = []
    if bn:            layers.append(nn.BatchNorm1d(n_input))
    if dropout > 0:   layers.append(nn.Dropout(dropout))
    layers.append(nn.Linear(n_input, n_classes))
    return nn.Sequential(*layers)


def fit_decoder(X_train, y_train, X_test, y_test, n_classes, variant,
                n_epochs=200, lr=1e-3, device="cpu"):
    """Train one decoder variant (full-batch Adam); return test accuracy = argmax of the logits."""
    if variant["zscore"]:
        mu, sd = X_train.mean(0), X_train.std(0) + 1e-6
        X_train, X_test = (X_train - mu) / sd, (X_test - mu) / sd
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long,    device=device)
    X_test_tensor  = torch.tensor(X_test,  dtype=torch.float32, device=device)
    y_test_tensor  = torch.tensor(y_test,  dtype=torch.long,    device=device)

    model     = build_decoder(X_train.shape[1], n_classes, variant["bn"], variant["dropout"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    if variant["loss"] == "ce":
        criterion, target = nn.CrossEntropyLoss(), y_train_tensor
    else:
        criterion, target = nn.MSELoss(), nn.functional.one_hot(y_train_tensor, n_classes).float()

    model.train()
    for _ in range(n_epochs):
        optimizer.zero_grad()
        criterion(model(X_train_tensor), target).backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        return (model(X_test_tensor).argmax(1) == y_test_tensor).float().mean().item()


In [ ]:
# --- NaN-free binning: keep only 50 ms bins with no NaN across ALL trials/neurons ---
def alignment_valid_mask(session_data_event):
    """Boolean (n_timebins,): True where NO trial/neuron is NaN across all sessions."""
    masks = []
    for session in session_data_event.values():
        masks.append(~np.isnan(session["spikes"]).any(axis=(0, 1)))  # (n_timebins,)
    return np.all(masks, axis=0) if masks else None


def compute_bins(alignment, valid_mask, bin_size=50):
    """Return [(t0,t1), ...] index-space bins that are fully NaN-free, and their centre times (ms)."""
    start_ms = GP_CONFIG[alignment]["start_time_ms"]
    T = len(valid_mask)
    edges, centres = [], []
    for t in range(0, T - bin_size + 1, bin_size):
        if valid_mask[t:t + bin_size].all():
            edges.append((t, t + bin_size))
            centres.append(start_ms + t + bin_size // 2)
    return edges, np.array(centres)

In [ ]:
# --- core: time-resolved pseudo-population decoding for one variable x one alignment ---
def decode_pseudo_population(session_data_event, label_col, label_values, neuron_positions,
                             n_total_neurons, bin_edges,
                             n_train=800, n_test=200, n_repeats=8, test_size=0.2,
                             n_epochs=200, lr=1e-3, variant=None, device="cpu", seed0=0,
                             n_shuffles=10):
    """Returns (repeat_accs (n_repeats, n_bins), mean_acc, sem_acc, null_accs (n_shuffles, n_bins)).

    real : n_repeats 80/20 real-trial splits (one split per repeat, reused across all bins) -> mean +/- SEM.
    null : n_shuffles WITHIN-SESSION label permutations, same decode -> full null_accs array; the plot
           z-tests real vs this shuffle distribution per bin (Fig 5b: thickness-coded significance bar).
    Sessions lacking any class are dropped (their columns stay 0 = constant = harmless).
    """
    n_classes = len(label_values)
    n_bins    = len(bin_edges)
    eligible_sessions = [session_id for session_id, session in session_data_event.items()
                         if all((session[label_col] == class_value).sum() > 0 for class_value in label_values)]

    def run_repeat(labels_by_session, rng):
        """One 80/20 split + bootstrap on the GIVEN labels, decode every bin.
        Returns (accs (n_bins,), models {bin_idx: state_dict} or None, (train_idx, test_idx))."""
        train_idx, test_idx = {}, {}
        for class_idx, class_value in enumerate(label_values):
            for session_id in eligible_sessions:
                n_trials = int((labels_by_session[session_id] == class_value).sum())
                perm = rng.permutation(n_trials)
                n_test_pool = max(1, int(n_trials * test_size))
                if n_test_pool >= n_trials:               # too few trials -> bootstrap all for both
                    train_pool = test_pool = np.arange(n_trials)
                else:
                    test_pool, train_pool = perm[:n_test_pool], perm[n_test_pool:]
                train_idx[(class_idx, session_id)] = rng.choice(train_pool, n_train, replace=True)
                test_idx[(class_idx, session_id)]  = rng.choice(test_pool,  n_test,  replace=True)
        accs = np.zeros(n_bins)
        for bin_idx, (t0, t1) in enumerate(bin_edges):
            X_train = np.zeros((n_classes * n_train, n_total_neurons))
            X_test  = np.zeros((n_classes * n_test,  n_total_neurons))
            for class_idx, class_value in enumerate(label_values):
                train_start, test_start = class_idx * n_train, class_idx * n_test
                for session_id in eligible_sessions:
                    class_mask = labels_by_session[session_id] == class_value
                    binned_rate = session_data_event[session_id]["spikes"][class_mask][:, :, t0:t1].mean(-1)
                    session_columns = neuron_positions[session_id]
                    X_train[train_start:train_start + n_train, session_columns] = binned_rate[train_idx[(class_idx, session_id)]]
                    X_test[test_start:test_start + n_test,     session_columns] = binned_rate[test_idx[(class_idx, session_id)]]
            y_train = np.repeat(np.arange(n_classes), n_train)
            y_test  = np.repeat(np.arange(n_classes), n_test)
            accs[bin_idx] = fit_decoder(X_train, y_train, X_test, y_test, n_classes, variant,
                                        n_epochs=n_epochs, lr=lr, device=device)
        return accs

    true_labels = {session_id: session_data_event[session_id][label_col] for session_id in eligible_sessions}

    # --- real decode ---
    repeat_accs = np.zeros((n_repeats, n_bins))
    for repeat_idx in range(n_repeats):
        rng = np.random.default_rng(seed0 + repeat_idx)
        repeat_accs[repeat_idx] = run_repeat(true_labels, rng)

    # --- shuffled null (within-session label permutation) ---
    null_accs = np.zeros((n_shuffles, n_bins))
    for shuffle_idx in range(n_shuffles):
        rng = np.random.default_rng(seed0 + 10_000 + shuffle_idx)
        shuffled_labels = {session_id: rng.permutation(true_labels[session_id]) for session_id in eligible_sessions}
        null_accs[shuffle_idx] = run_repeat(shuffled_labels, rng)

    return repeat_accs, repeat_accs.mean(0), repeat_accs.std(0) / np.sqrt(n_repeats), null_accs

In [ ]:
# --- hyperparameters ---
N_TRAIN, N_TEST, N_REPEATS = 800, 200, 8   # per class, as 8.12
BIN_SIZE  = 50
N_EPOCHS  = 200
LR        = 1e-3
N_SHUFFLES = 3       # reduced from 8.12's 10: the ablation compares real-decode accuracy; null only sanity-checks chance
FORCE_RERUN = False
decode_variables = ["abs_coherence", "choice"]
block_conditions = {"equal": 0, "unequal": 1}


In [ ]:
# --- decode (or reload): {variant} x {block_condition} x {direction} x {variable} x {alignment} ---
decoder_dir = processed_dir / "decoder"
decoder_dir.mkdir(parents=True, exist_ok=True)
results_path = decoder_dir / "decoding_results_8.13.pkl"
if not FORCE_RERUN and results_path.exists():
    with open(results_path, "rb") as f:
        results = pickle.load(f)
    print(f"loaded existing {results_path.name} (set FORCE_RERUN=True to recompute)")
else:
    if "ephys_data" in globals():        # 3 GB off a slow share: never re-read within a live kernel
        print("reusing ephys_data already in memory")
    else:
        print("loading ephys_neuron_wise.pkl (3 GB, slow network share) ...")
        with open(processed_dir / "ephys_neuron_wise.pkl", "rb") as f:
            ephys_data = pickle.load(f)
    results = {name: {} for name in VARIANTS}
    for block_condition, block_value in block_conditions.items():
        block_trial_info = trial_info[trial_info.prior_block == block_value]
        for name in VARIANTS:
            results[name][block_condition] = {}
        for direction in ["toRF", "awayRF"]:
            direction_trial_info = block_trial_info[block_trial_info.prior_direction == direction].reset_index(drop=True)
            sessions = sorted(direction_trial_info.session_id.unique())
            session_data, neuron_positions, n_total_neurons, label_values = decoding_utils.get_session_trial_data(
                sessions, direction_trial_info, neuron_metadata, ephys_data, alignments)
            print(f"\n=== {block_condition} block | {direction}: {len(direction_trial_info)} trials, {n_total_neurons} neurons ===")
            for name in VARIANTS:
                results[name][block_condition][direction] = {"n_total_neurons": n_total_neurons, "label_values": label_values, "decode": {}}
            for variable in decode_variables:
                for name, variant in VARIANTS.items():
                    results[name][block_condition][direction]["decode"][variable] = {}
                    for alignment in alignments:
                        valid_mask = alignment_valid_mask(session_data[alignment])
                        edges, centres = compute_bins(alignment, valid_mask, BIN_SIZE)
                        _, mean_acc, sem, null_accs = decode_pseudo_population(
                            session_data[alignment], variable, label_values[variable], neuron_positions, n_total_neurons, edges,
                            n_train=N_TRAIN, n_test=N_TEST, n_repeats=N_REPEATS, n_shuffles=N_SHUFFLES,
                            n_epochs=N_EPOCHS, lr=LR, variant=variant, device=str(device))
                        results[name][block_condition][direction]["decode"][variable][alignment] = {
                            "centres": centres, "mean": mean_acc, "sem": sem, "null": null_accs}
                    print(f"  {variable} | {name}: done")
    with open(results_path, "wb") as f:
        pickle.dump(results, f)
    print(f"\nsaved {results_path.name}")


In [ ]:
# --- plot: variants overlaid; one figure per block x direction x variable ---
epoch_titles = {"visual": "visual target", "cue": "cue", "response": "response"}
colors = dict(zip(VARIANTS, ["black", "tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]))
for block_condition in block_conditions:
    for direction in ["toRF", "awayRF"]:
        for variable in decode_variables:
            chance = 1.0 / len(results["full"][block_condition][direction]["label_values"][variable])
            aligns = [al for al in alignments if al != "baseline"]
            fig, axes = plt.subplots(1, len(aligns), figsize=(11, 3), sharey=True)
            for ax, alignment in zip(axes, aligns):
                ax.axhline(chance, ls=":", c="lightgray", lw=0.8)
                for name in VARIANTS:
                    d = results[name][block_condition][direction]["decode"][variable][alignment]
                    ax.plot(d["centres"], d["mean"], color=colors[name], lw=1.2, label=name)
                    ax.fill_between(d["centres"], d["mean"] - d["sem"], d["mean"] + d["sem"], color=colors[name], alpha=0.12)
                ax.axvline(0, c="k", lw=0.5, ls="--")
                ax.set_title(epoch_titles[alignment], fontsize=9); ax.set_xlabel("time (ms)")
            axes[0].set_ylabel("accuracy"); axes[-1].legend(fontsize=7, frameon=False)
            fig.suptitle(f"{block_condition} block | {direction} | {variable}", fontsize=10)
            plt.tight_layout(); plt.show()

# summary: peak-accuracy difference vs `full` (mean over bins of |Δ| and max Δ), per variant
rows = []
for name in VARIANTS:
    diffs = []
    for bc in block_conditions:
        for dr in ["toRF", "awayRF"]:
            for var in decode_variables:
                for al in alignments:
                    full = results["full"][bc][dr]["decode"][var][al]["mean"]
                    this = results[name][bc][dr]["decode"][var][al]["mean"]
                    diffs.append(this - full)
    diffs = np.concatenate(diffs)
    rows.append({"variant": name, "mean_diff_vs_full": diffs.mean(), "mean_abs_diff": np.abs(diffs).mean(), "worst_diff": diffs.min()})
print(pd.DataFrame(rows).round(4).to_string(index=False))
